# LLM Agent Pipeline Tuning Notebook

This notebook is a lightweight, local version of a LangChain/LangGraph-style agent workflow for circadian medicine reports. It keeps the workflow simple: every node is still a normal notebook cell, but the cells share one explicit `state` object and each agent has a clear contract.

**Goal**
Tune prompts, retrieval settings, and local Ollama model choices before copying the final prompts back into `tools/llm_conversation.py`.

**Pipeline graph**

```text
period-comparison JSON from DB
    -> Agent 1: data summariser
    -> Agent 2: PubMed query planner
    -> Agent 3a: deterministic PubMed retrieval
    -> Agent 3b: LLM relevance judge
    -> Agent 4: literature synthesiser
    -> Agent 6: symptom-metric linker (optional)
    -> Agent 5: audience-aware report writer
    -> prompt diff against production source
```

**How to use this notebook**
1. Run imports, configuration, and DB loading cells once.
2. Edit `AGENT_CONFIG` to choose the local model for each LLM agent.
3. Run agents in order. Each agent reads from and writes to `state`.
4. Tune one prompt or one model at a time, then re-run that agent and downstream cells.
5. Use the final diff cell to copy stable prompts back into `tools/llm_conversation.py`.

**LangChain/LangGraph mapping**
- `state` is the graph state.
- Each agent section is a graph node.
- PubMed search is a deterministic tool node.
- `AGENT_CONFIG` is the local model/router configuration.
- The final diff cell is the evaluation checkpoint for prompt changes.


In [1]:
import sys, json, sqlite3, time, subprocess, difflib
from pathlib import Path

# Walk up from cwd to find project root (the dir containing tools/llm_conversation.py).
PROJECT_ROOT = Path.cwd()
for _ in range(6):
    if (PROJECT_ROOT / "tools" / "llm_conversation.py").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Could not locate project root containing tools/llm_conversation.py")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tools.llm_conversation as _lc
from tools.llm_conversation import (
    _chat,
    _compact_report_for_llm,
    _parse_structured_queries,
    _llm_keep_pmids,
    _sanitize_pmids,
)
from tools.pubmed_search import search_pubmed, evidence_to_text, RetrievalConfig

print("Imports OK")
print("Project root:", PROJECT_ROOT)

# Shared mutable state. Each cell reads/writes here.
state = {
    "compact_report": "",
    "data_summary": "",
    "search_queries": [],
    "raw_abstracts": "",
    "pmid_list": [],
    "evidence_items": [],
    "lit_summary": "",
    "symptom_metric_table": "",
    "claim_to_pmid_map": {},
    "final_report": "",
}

Imports OK
Project root: /Users/arahjou/Documents/APP_CIRCADIAN_MEDICINE_v7


/Users/arahjou/Documents/Com_Conda/.conda/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 1. Configuration: Local Models and Graph Contracts

Choose the audience, optional anamnesis, and the local Ollama model assigned to each LLM agent.

`AGENT_CONFIG` is the notebook's simplified LangGraph-style control panel:
- **node**: the graph step name
- **model**: the local model used by that agent
- **input**: the `state` key or tool input the node reads
- **output**: the `state` key the node writes
- **tune**: what usually changes during prompt/model tuning

Agent 3a has no model because it is deterministic PubMed retrieval. Agent 2 can use a fast model first and a fallback model if JSON parsing or generation fails.


In [2]:
DB_PATH = PROJECT_ROOT / "Actigraph_record.db"
AUDIENCE = "doctor"          # "expert" | "doctor" | "layperson"
ANAMNESIS = ""               # set to non-empty text to activate Agent 6

AGENT_CONFIG = {
    "agent1": {
        "node": "summarise_actigraphy_data",
        "model": "phi4:14b",
        "input": "state['compact_report']",
        "output": "state['data_summary']",
        "tune": "clinical specificity, abnormal metrics, concise summary",
    },
    "agent2": {
        "node": "plan_pubmed_queries",
        "model": "llama3.2:latest",
        "fallback_model": "phi4:14b",
        "input": "state['data_summary'] + optional ANAMNESIS",
        "output": "state['search_queries']",
        "tune": "MeSH-friendly terms, query breadth, JSON reliability",
    },
    "agent3a": {
        "node": "retrieve_pubmed_evidence",
        "model": None,
        "input": "state['search_queries']",
        "output": "state['evidence_items'], state['pmid_list']",
        "tune": "retmax, years_back, humans/adults filters",
    },
    "agent3b": {
        "node": "judge_evidence_relevance",
        "model": "qwen3.5:latest",
        "input": "state['evidence_items'], state['data_summary']",
        "output": "filtered state['evidence_items']",
        "tune": "keep/drop criteria, schema-following model choice",
    },
    "agent4": {
        "node": "synthesise_literature",
        "model": "phi4:14b",
        "input": "state['data_summary'], state['raw_abstracts']",
        "output": "state['lit_summary']",
        "tune": "evidence linkage, length, neutral wording",
    },
    "agent6": {
        "node": "link_symptoms_to_metrics",
        "model": "phi4:14b",
        "input": "ANAMNESIS, state['data_summary'], state['raw_abstracts']",
        "output": "state['symptom_metric_table']",
        "tune": "symptom coverage, metric matching, PMID support",
    },
    "agent5": {
        "node": "write_audience_report",
        "model": "phi4:14b",
        "input": "state['data_summary'], state['lit_summary'], optional Agent 6 output",
        "output": "state['final_report']",
        "tune": "audience fit, structure, citation discipline",
    },
}

# Backward-compatible model lookup used by the existing cells.
MODELS = {
    "agent1": AGENT_CONFIG["agent1"]["model"],
    "agent2_fast": AGENT_CONFIG["agent2"]["model"],
    "agent2_fallback": AGENT_CONFIG["agent2"]["fallback_model"],
    "relevance_judge": AGENT_CONFIG["agent3b"]["model"],
    "agent4": AGENT_CONFIG["agent4"]["model"],
    "agent5": AGENT_CONFIG["agent5"]["model"],
    "agent6": AGENT_CONFIG["agent6"]["model"],
}

def print_agent_config():
    print("Graph nodes and local models:\n")
    for agent, cfg in AGENT_CONFIG.items():
        model = cfg["model"] or "deterministic tool"
        fallback = f" (fallback: {cfg['fallback_model']})" if cfg.get("fallback_model") else ""
        print(f"{agent:7} | {cfg['node']:28} | {model}{fallback}")
        print(f"        input : {cfg['input']}")
        print(f"        output: {cfg['output']}")
        print(f"        tune  : {cfg['tune']}\n")

print_agent_config()
print("Available Ollama models:")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)


Graph nodes and local models:

agent1  | summarise_actigraphy_data    | phi4:14b
        input : state['compact_report']
        output: state['data_summary']
        tune  : clinical specificity, abnormal metrics, concise summary

agent2  | plan_pubmed_queries          | llama3.2:latest (fallback: phi4:14b)
        input : state['data_summary'] + optional ANAMNESIS
        output: state['search_queries']
        tune  : MeSH-friendly terms, query breadth, JSON reliability

agent3a | retrieve_pubmed_evidence     | deterministic tool
        input : state['search_queries']
        output: state['evidence_items'], state['pmid_list']
        tune  : retmax, years_back, humans/adults filters

agent3b | judge_evidence_relevance     | qwen3.5:latest
        input : state['evidence_items'], state['data_summary']
        output: filtered state['evidence_items']
        tune  : keep/drop criteria, schema-following model choice

agent4  | synthesise_literature        | phi4:14b
        input : s

## 2. Load period-comparison JSON from DB

Reads `ai_analysis_runs.json_input` for the chosen row and flattens it into the
compact text format that Agent 1 expects.

In [3]:
con = sqlite3.connect(DB_PATH)
rows = con.execute(
    "SELECT username, period_id_1, period_id_2, audience, model, created_at "
    "FROM ai_analysis_runs ORDER BY created_at DESC"
).fetchall()
print(f"Available rows: {len(rows)}")
for r in rows[:20]:
    print(f"  username={r[0]:<10}  P1={r[1]:<10}  P2={r[2]:<10}  audience={r[3]:<10}  model={r[4]:<14}  created={r[5]}")

# --- pick one (edit me) ---
ROW_KEY = (rows[0][0], rows[0][1], rows[0][2])  # most recent run by default
# ROW_KEY = ("admin", "ID-001", "ID-002")
print(f"\nSelected: username={ROW_KEY[0]}  P1={ROW_KEY[1]}  P2={ROW_KEY[2]}")

raw = con.execute(
    "SELECT json_input FROM ai_analysis_runs "
    "WHERE username=? AND period_id_1=? AND period_id_2=? "
    "ORDER BY created_at DESC LIMIT 1",
    ROW_KEY,
).fetchone()
con.close()
if not raw or not raw[0]:
    raise RuntimeError(f"No json_input column for {ROW_KEY}")

report_data = json.loads(raw[0])
state["compact_report"] = _compact_report_for_llm(report_data)
print(f"\n--- compact_report ({len(state['compact_report'])} chars, head) ---")
print(state["compact_report"][:1500])

Available rows: 10
  username=admin       P1=ID-001      P2=ID-002      audience=doctor      model=qwen3.5:9b      created=2026-05-09 15:41:02
  username=admin       P1=ID-001      P2=ID-002      audience=doctor      model=phi4:14b        created=2026-05-09 14:36:51
  username=admin       P1=ID-001      P2=ID-002      audience=layperson   model=phi4:14b        created=2026-05-09 14:31:49
  username=admin       P1=D01_01      P2=D01_02      audience=doctor      model=qwen3.5:9b      created=2026-05-09 14:19:58
  username=admin       P1=D01_01      P2=D01_02      audience=expert      model=phi4:14b        created=2026-04-02 11:48:19
  username=admin       P1=D01_01      P2=D01_02      audience=doctor      model=phi4:14b        created=2026-04-02 11:42:42
  username=admin       P1=D01_01      P2=D01_02      audience=layperson   model=phi4:14b        created=2026-04-02 11:33:28
  username=user1       P1=D01_01      P2=D01_02      audience=expert      model=phi4:14b        created=2026-03-2

## 3. Agent 1 — Data Summariser

**Graph node:** `summarise_actigraphy_data`  
**Local model:** `AGENT_CONFIG["agent1"]["model"]`  
**Reads:** `state["compact_report"]`  
**Writes:** `state["data_summary"]`

**Purpose**
Compress the raw period-comparison metrics into a clinically useful summary for downstream retrieval and report writing.

**Tune when**
The summary loses important numeric changes, over-explains normal values, or uses language that is too broad for literature retrieval.


In [5]:
AGENT1_SYSTEM = """You are a circadian medicine data analyst.
Your job is to produce a concise clinical summary of the provided actigraphy metrics for a
downstream literature researcher who needs to find relevant evidence.

Rules:
- Use clinical language (IS, IV, RA, CPD, cosinor amplitude/acrophase, WASO, SRI, etc.).
- Highlight abnormal values, notable trends, and comparisons between Period 1 and Period 2.
- Be specific with numbers and include units where available.
- Do NOT interpret for a lay audience — this is for the literature search step.
- Limit your response to approximately 400 words."""

t0 = time.time()
state["data_summary"] = _chat(
    model_name=MODELS["agent1"],
    system=AGENT1_SYSTEM,
    user=f"# Actigraphy Metrics\n{state['compact_report']}\n\n"
         "Summarise these metrics for a downstream literature researcher.",
    temperature=0.2,
)
print(f"[{time.time()-t0:.1f}s] Agent 1 ({MODELS['agent1']}) → data_summary ({len(state['data_summary'])} chars):\n")
print(state["data_summary"])

[80.3s] Agent 1 (phi4:14b) → data_summary (2533 chars):

**Clinical Summary of Actigraphy Metrics**

**Activity Patterns:**
- **Mesor:** Period 1 (P1) exhibited a mesor of 37.39 compared to 35.67 in Period 2 (P2), indicating a decrease of approximately 1.72 units.
- **Cosinor Acrophase (CPD2):** No change was observed between P1 and P2, with both periods showing a CPD2 value of 1.29.
- **Interdaily Stability (IS):** Slight increase from 0.21 in P1 to 0.22 in P2 (Δ=0.01), suggesting improved stability across days.
- **Intradaily Variability (IV):** Remained constant at 0.52 for both periods, indicating no change in variability within the day.
- **Most Active 10h (M10):** Decreased from 53.57 in P1 to 50.27 in P2 (Δ=-3.30).
- **Least Active 5h (L5):** Reduced from 8.79 in P1 to 7.05 in P2 (Δ=-1.74).
- **Relative Amplitude (RA):** Consistent at 0.78 for both periods, showing no change.

**Light Exposure Patterns:**
- **Mesor:** Significant increase from 22.87 in P1 to 48.9 in P2 (Δ=26.03)

## 4. Agent 2 — PubMed Query Planner

**Graph node:** `plan_pubmed_queries`  
**Local model:** `AGENT_CONFIG["agent2"]["model"]` with `fallback_model`  
**Reads:** `state["data_summary"]` and optional `ANAMNESIS`  
**Writes:** `state["search_queries"]`

**Purpose**
Convert the clinical summary into compact PubMed query intents.

**Tune when**
Queries are too generic, miss chronobiology terms, produce duplicate topics, or fail to parse as JSON lines.


In [6]:
AGENT2_SYSTEM = """You are a biomedical literature search specialist.
Given a clinical summary of actigraphy data, extract PubMed query intents.

Rules:
- Output EXACTLY one compact JSON object per line with keys:
  {"topic":"...", "population":"...", "context":"...", "expected_link":"..."}
- Use MeSH-friendly terms in topic/context.
- Focus on the most abnormal or clinically notable findings from the summary.
- Each object should target a distinct aspect.
- topic should be 2–7 words, context 1–6 words, population typically humans/adults.
- Output ONLY JSON lines, no markdown, no commentary."""

AGENT2_ANAMNESIS_ADDENDUM = """
Additional context — Patient anamnesis (reported symptoms/history):
{anamnesis}

Because an anamnesis is available, generate a MIXED query set:
- 3 query objects focused on the most abnormal circadian/sleep metrics
- 2 additional query objects that combine a symptom with a metric change
Total: exactly 5 JSON lines."""

base_user = f"# Clinical Data Summary\n{state['data_summary']}\n\nOutput JSON lines query intents."
if ANAMNESIS.strip():
    base_user = (
        f"# Clinical Data Summary\n{state['data_summary']}\n"
        + AGENT2_ANAMNESIS_ADDENDUM.format(anamnesis=ANAMNESIS)
    )

t0 = time.time()
try:
    raw_text = _chat(MODELS["agent2_fast"], AGENT2_SYSTEM, base_user, temperature=0.1)
    used = MODELS["agent2_fast"]
except Exception as e:
    print(f"  ↳ fast model {MODELS['agent2_fast']} failed ({e!r}), falling back to {MODELS['agent2_fallback']}")
    raw_text = _chat(MODELS["agent2_fallback"], AGENT2_SYSTEM, base_user, temperature=0.1)
    used = MODELS["agent2_fallback"]

state["search_queries"] = _parse_structured_queries(raw_text)
print(f"[{time.time()-t0:.1f}s] Agent 2 ({used}) → {len(state['search_queries'])} parsed queries")
print("\n--- raw output ---")
print(raw_text)
print("\n--- parsed queries ---")
for q in state["search_queries"]:
    print(" -", json.dumps(q, ensure_ascii=False))

[4.9s] Agent 2 (llama3.2:latest) → 3 parsed queries

--- raw output ---
{"topic":"Light Exposure","population":"Adults","context":"Increased mesor and M10 values indicate higher overall and peak light exposure","expected_link":"https://www.ncbi.nlm.nih.gov/pubmed/"} 

{"topic":"Sleep Duration","population":"Adults","context":"Marked decrease from 266.0 minutes to 16.0 minutes between periods","expected_link":"https://www.ncbi.nlm.nih.gov/pubmed/"} 

{"topic":"Social Jetlag Index","population":"Adults","context":"Improved from 84.34 to 90.83 with reduced social jetlag","expected_link":"https://www.ncbi.nlm.nih.gov/pubmed/"}

--- parsed queries ---
 - {"topic": "Light Exposure", "population": "Adults", "context": "Increased mesor and M10 values indicate higher overall and peak light exposure", "expected_link": "https://www.ncbi.nlm.nih.gov/pubmed/"}
 - {"topic": "Sleep Duration", "population": "Adults", "context": "Marked decrease from 266.0 minutes to 16.0 minutes between periods", "exp

## 5. Agent 3a — PubMed Retrieval Tool

**Graph node:** `retrieve_pubmed_evidence`  
**Local model:** none; deterministic tool node  
**Reads:** `state["search_queries"]`  
**Writes:** `state["evidence_items"]`, `state["pmid_list"]`

**Purpose**
Run NCBI/PubMed retrieval from the query intents. This is the retrieval/tool step in LangChain terms.

**Tune when**
Retrieval returns too few papers, too many generic papers, or papers outside the patient-relevant time/population window.


In [7]:
RETRIEVAL_CFG = RetrievalConfig(
    retmax_per_query=15,
    keep_per_query=5,
    max_total_items=18,
    years_back=15,                         # widen to find more
    humans_only=True,
    adults_only=(AUDIENCE == "doctor"),    # adds adult[MeSH] filter
)

t0 = time.time()
state["evidence_items"] = search_pubmed(state["search_queries"], config=RETRIEVAL_CFG)
state["pmid_list"] = [str(x.get("pmid")) for x in state["evidence_items"] if x.get("pmid")]
state["raw_abstracts"] = evidence_to_text(state["evidence_items"])
print(f"[{time.time()-t0:.1f}s] Agent 3a → {len(state['evidence_items'])} items")
print()
for ev in state["evidence_items"]:
    pmid = ev.get("pmid", "?")
    year = ev.get("year", "?")
    score = ev.get("score", 0.0)
    title = (ev.get("title") or "")[:90]
    print(f"  PMID {pmid:>10}  ({year})  score={score:.3f}  | {title}")

[7.6s] Agent 3a → 15 items

  PMID   33558966  (2021)  score=0.360  | Effects of sleep deprivation on endothelial function in adult humans: a systematic review.
  PMID   28525634  (2017)  score=0.360  | Association Between Weekend Catch-up Sleep and Lower Body Mass: Population-Based Study.
  PMID   27318597  (2016)  score=0.360  | Influence of lithium on sleep and chronotypes in remitted patients with bipolar disorder.
  PMID   22578422  (2012)  score=0.333  | Social jetlag and obesity.
  PMID   30663440  (2019)  score=0.333  | Relationships between chronotype, social jetlag, sleep, obesity and blood pressure in heal
  PMID   38353253  (2024)  score=0.333  | Patterns in behavioural sleep variables and social jetlag in elderly people of Western Odi
  PMID   36827078  (2023)  score=0.333  | Population-representative study reveals cardiovascular and metabolic disease biomarkers as
  PMID   34520928  (2022)  score=0.320  | Sex-specific association of the lunar cycle with sleep.
  PMID   35

## 6. Agent 3b — Relevance Judge

**Graph node:** `judge_evidence_relevance`  
**Local model:** `AGENT_CONFIG["agent3b"]["model"]`  
**Reads:** `state["evidence_items"]`, `state["data_summary"]`, `state["search_queries"]`  
**Writes:** filtered `state["evidence_items"]`, `state["raw_abstracts"]`, `state["pmid_list"]`

**Purpose**
Drop off-topic retrieved papers before synthesis.

**Tune when**
The judge keeps broad sleep papers, drops directly relevant circadian papers, or fails to return valid JSON.


In [8]:
RELEVANCE_SYSTEM = """You are a biomedical relevance filter.
Input:
1) structured PubMed evidence items (PMID + title + abstract + lexical score)
2) the clinical summary and query intents

Output a single JSON object with this exact shape:
  {"keep_pmids": ["12345678", "23456789", ...]}

Rules:
- keep_pmids must contain ONLY PMIDs from the input evidence items, as strings.
- Keep evidence directly relevant to circadian rhythm, sleep regularity, actigraphy,
  light exposure, chronobiology, or sleep-wake patterns.
- Prefer higher-quality clinical relevance over broad background biology.
- Drop papers that are off-topic for the queries (e.g. ultra-endurance athletes,
  professional tennis players, tangential physiology) UNLESS the clinical summary
  specifically calls them out.
- If none are relevant, return {"keep_pmids": []}.
- Output JSON only, no prose, no markdown."""

# _llm_keep_pmids() reads tools.llm_conversation._RELEVANCE_SYSTEM. Swap our
# notebook-edited prompt in temporarily so changes here take effect.
prev_prompt = _lc._RELEVANCE_SYSTEM
_lc._RELEVANCE_SYSTEM = RELEVANCE_SYSTEM
t0 = time.time()
try:
    kept = _llm_keep_pmids(
        evidence_items=state["evidence_items"],
        queries=state["search_queries"],
        summary=state["data_summary"],
        fast_model=MODELS["agent2_fast"],
        fallback_model=MODELS["relevance_judge"],
    )
finally:
    _lc._RELEVANCE_SYSTEM = prev_prompt

print(f"[{time.time()-t0:.1f}s] Agent 3b → judge kept {len(kept)} of {len(state['evidence_items'])}")
print()
print("Decisions:")
for ev in state["evidence_items"]:
    mark = "✓" if str(ev.get("pmid")) in kept else "✗"
    print(f"  {mark} PMID {ev.get('pmid')}: {(ev.get('title') or '')[:90]}")

if kept:
    state["evidence_items"] = [x for x in state["evidence_items"] if str(x.get("pmid")) in kept]
    state["pmid_list"]      = [str(x.get("pmid")) for x in state["evidence_items"] if x.get("pmid")]
    state["raw_abstracts"]  = evidence_to_text(state["evidence_items"])
    print(f"\nFiltered → {len(state['evidence_items'])} items kept")
else:
    print("\nJudge returned empty set — keeping all items (no filter applied).")

KeyboardInterrupt: 

## 7. Agent 4 — Literature Synthesiser

**Graph node:** `synthesise_literature`  
**Local model:** `AGENT_CONFIG["agent4"]["model"]`  
**Reads:** `state["data_summary"]`, `state["raw_abstracts"]`  
**Writes:** `state["lit_summary"]`

**Purpose**
Turn kept abstracts into a short evidence summary that can support the final report.

**Tune when**
The synthesis is too long, fails to connect evidence to patient metrics, or makes claims stronger than the abstracts support.


In [ ]:
AGENT4_SYSTEM = """You are a scientific summariser for a medical evidence review.
You will receive a set of PubMed abstracts and a clinical data summary.

Your task:
- Identify findings in the abstracts that are relevant to the patient's circadian patterns.
- Extract: key associations, risk factors, and recommendations that are evidence-supported.
- Explicitly link each finding back to specific abnormal metrics mentioned in the data summary.
- Be concise: 200 words maximum.
- Use neutral scientific language. Do not over-interpret."""

user_content = (
    f"# Clinical Data Summary\n{state['data_summary']}\n\n"
    f"# PubMed Abstracts\n{state['raw_abstracts']}\n\n"
    "Identify relevant findings and produce a 200-word evidence summary."
)
t0 = time.time()
state["lit_summary"] = _chat(MODELS["agent4"], AGENT4_SYSTEM, user_content, temperature=0.2)
print(f"[{time.time()-t0:.1f}s] Agent 4 ({MODELS['agent4']}) → lit_summary ({len(state['lit_summary'])} chars):\n")
print(state["lit_summary"])

## 8. Agent 6 — Symptom-Metric Linker Optional Node

**Graph node:** `link_symptoms_to_metrics`  
**Local model:** `AGENT_CONFIG["agent6"]["model"]`  
**Reads:** `ANAMNESIS`, `state["data_summary"]`, `state["raw_abstracts"]`  
**Writes:** `state["symptom_metric_table"]`

**Purpose**
When anamnesis text is available, map reported symptoms to actigraphy changes and literature support.

**Tune when**
Symptoms are skipped, metric links are vague, or PMID support is missing where evidence exists.


In [ ]:
AGENT6_SYSTEM = """You are a clinical evidence mapper for a circadian medicine report.
You will receive:
1. A patient anamnesis (reported symptoms and medical history)
2. A clinical summary of actigraphy metric changes between two periods
3. PubMed abstracts retrieved for this patient

Your task: for EACH symptom or complaint reported in the anamnesis, produce ONE row in a
structured table that maps:
  Symptom → Most relevant metric change (Δ) → Literature support (PMID if available)

Output format — use EXACTLY this layout, no extra commentary:

Symptom-Metric Correlation Table
|Symptom|Relevant Metric Change|Evidence Support|PMID|
|---|---|---|---|
|<symptom>|<metric: direction + value>|<1-sentence finding from abstract, or \"No direct evidence found\">|<PMID or —>|

After the table, add a short section:
\"Unexplained symptoms (no supporting literature found):\"
- List any symptom where no abstract contained relevant evidence.
- If all symptoms are supported, write: All reported symptoms have supporting literature.

Rules:
- If a symptom is not clearly linked to any metric change, say so explicitly.
- Do NOT diagnose. Use cautious language.
- Be concise."""

if not ANAMNESIS.strip():
    print("ANAMNESIS is empty → Agent 6 skipped (set ANAMNESIS in the config cell to enable).")
else:
    user_content = (
        f"# Patient Anamnesis\n{ANAMNESIS}\n\n"
        f"# Clinical Data Summary (metric changes)\n{state['data_summary']}\n\n"
        f"# PubMed Abstracts\n{state['raw_abstracts']}\n\n"
        "Produce the Symptom-Metric Correlation Table as instructed."
    )
    t0 = time.time()
    state["symptom_metric_table"] = _chat(MODELS["agent6"], AGENT6_SYSTEM, user_content, temperature=0.1)
    print(f"[{time.time()-t0:.1f}s] Agent 6 ({MODELS['agent6']}) → symptom_metric_table:\n")
    print(state["symptom_metric_table"])

## 9. Agent 5 — Audience-Aware Report Writer

**Graph node:** `write_audience_report`  
**Local model:** `AGENT_CONFIG["agent5"]["model"]`  
**Reads:** `state["data_summary"]`, `state["lit_summary"]`, optional `state["symptom_metric_table"]`  
**Writes:** `state["final_report"]`

**Purpose**
Create the final report for `AUDIENCE` while preserving citation discipline.

**Tune when**
The report does not match the audience, loses key metrics, overstates evidence, or hallucinates citations.


In [ ]:
AGENT5_AUDIENCES = {
    "expert": """You are a circadian medicine expert writing a technical report for a peer specialist.

Use technical language: IS, IV, RA, CPD, cosinor amplitude/acrophase/mesor, WASO, SRI, PSG,
DLMO, zeitgeber, phase angle, interdaily stability, relative amplitude, melanopic EDI, etc.
Include specific metric values and Δ changes.
Reference the relevant literature findings where applicable.

Required output structure (use these exact headings):

Summary: <2–3 sentences: key findings and overall circadian phenotype>

1) Circadian Rhythm Parameters:
- <IS, IV, RA values with interpretations>

2) Sleep Architecture Indicators:
- <onset, offset, mid-sleep, WASO, SRI findings>

3) Light Exposure Analysis:
- <daytime/nocturnal melanopic EDI, phase alignment implications>

4) Phase & Cosinor Analysis:
- <acrophase, amplitude, mesor, CPD values and clinical relevance>

5) Literature-Supported Associations:
- <evidence-based links to risk factors or conditions>

6) Clinical Recommendations:
- <targeted, evidence-based interventions with mechanism>""",

    "doctor": """You are a clinician writing a circadian health report for a referring physician.

Use clinical language appropriate for a medical doctor: refer to sleep-wake patterns,
circadian misalignment, fragmentation, light hygiene, metabolic and cardiovascular risk
associations. Avoid deep chronobiology jargon but do use standard clinical terms.
Include ICD-adjacent language where appropriate (e.g., insomnia, hypersomnia, circadian
rhythm sleep-wake disorder).

Required output structure (use these exact headings):

Summary: <1–2 sentences: overall clinical impression>

1) Key Findings:
- <3–6 bullets: abnormal or notable metrics in clinical terms>

2) Symptom & Risk Associations (evidence-supported):
- <3–5 bullets: pattern → likely symptoms → medium/long-term risks>

3) Relevant Literature:
- <2–4 bullets citing key evidence from the abstracts>

4) Management Recommendations:
- <3–5 bullets: practical, timed interventions; note any referral indications>

Safety note: <1 sentence>""",

    "layperson": """You are a circadian health coach writing a plain-language report for a patient.

Rules:
- Plain language. No acronyms, no jargon. If you must mention a term, explain it in plain words.
- Do NOT diagnose. Use cautious language: \"can be linked to\", \"may affect\", \"is often associated with\".
- Use short sentences, bullet points, and everyday analogies.
- Be encouraging and practical.

Required output structure (use these exact headings):

Summary: <1–2 sentences: plain-English overall picture>

1) What looks off (in simple terms):
- <2–5 bullets>

2) What looks good — keep doing this:
- <2–5 bullets>

3) Your 7-day action plan:
- <4–6 bullets, each starts with a verb, include approximate timing>

4) How these patterns can affect you:
- <3–5 bullets: pattern → everyday symptom → longer-term risk, in simple words>

Safety note: <1 sentence encouraging professional help if symptoms are significant>""",
}

system_prompt = AGENT5_AUDIENCES.get(AUDIENCE, AGENT5_AUDIENCES["layperson"])
evidence_items = state["evidence_items"]
has_evidence = bool(evidence_items)

user_content = (
    f"# Actigraphy Data Context\n{state['compact_report']}\n\n"
    f"# Clinical Data Summary\n{state['data_summary']}\n\n"
    f"# Supporting Literature Evidence\n{state['lit_summary']}\n\n"
    f"# Evidence Items (use PMIDs from here only)\n{json.dumps(evidence_items, ensure_ascii=False)}\n\n"
)
if state["symptom_metric_table"].strip():
    user_content += (
        f"# Symptom-Metric Correlation (Agent 6 output)\n{state['symptom_metric_table']}\n\n"
        "When writing the report, include a section titled 'Symptom-Metric Correlation' "
        "that incorporates the table above and highlights any unexplained symptoms.\n\n"
    )
if has_evidence:
    user_content += (
        "Write the final report now.\n"
        "Citation rules — STRICT:\n"
        "- Every evidence-backed claim must cite a real PMID in parentheses, e.g. (PMID 12345678).\n"
        "- ONLY use PMIDs that appear in the Evidence Items list above. Never invent or guess a PMID.\n"
        "- NEVER write a bare placeholder like '(PMID)' with no number. If you have no PMID for a claim, "
        "write '(no direct evidence found)' instead, or omit the claim."
    )
else:
    user_content += (
        "Write the final report now.\n"
        "IMPORTANT — no PubMed evidence was retrieved for this case:\n"
        "- Do NOT cite any PMIDs anywhere in the report.\n"
        "- Do NOT write '(PMID)', '(PMID: ...)', or any placeholder suggesting a citation.\n"
        "- For any claim that would normally need literature support, either omit it or label it as "
        "'(no direct evidence found)'.\n"
        "- You may still report data-driven findings from the Clinical Data Summary."
    )

t0 = time.time()
final_report = _chat(MODELS["agent5"], system_prompt, user_content, temperature=0.2)
allowed = {str(p) for p in state["pmid_list"]}
final_report = _sanitize_pmids(final_report, allowed)

if state["pmid_list"]:
    ref_lines = ["\n\n---\nReferences (PubMed):"]
    for i, pmid in enumerate(state["pmid_list"], start=1):
        ref_lines.append(f"{i}. PMID {pmid} — https://pubmed.ncbi.nlm.nih.gov/{pmid}/")
    final_report = final_report + "\n".join(ref_lines)

state["final_report"] = final_report
print(f"[{time.time()-t0:.1f}s] Agent 5 ({MODELS['agent5']}) → final report ({len(final_report)} chars):\n")
print(final_report)

## 10. Evaluation Checkpoint — Diff vs Source

This is the notebook's simple evaluation checkpoint. It shows which tuned prompts differ from the live constants in `tools/llm_conversation.py`.

Use this cell after a tuning pass to decide which prompt changes are stable enough to move back into production code.


In [ ]:
PROMPTS_TO_DIFF = [
    ("AGENT1_SYSTEM",                _lc._AGENT1_SYSTEM,                 AGENT1_SYSTEM),
    ("AGENT2_SYSTEM",                _lc._AGENT2_SYSTEM,                 AGENT2_SYSTEM),
    ("AGENT4_SYSTEM",                _lc._AGENT4_SYSTEM,                 AGENT4_SYSTEM),
    ("AGENT6_SYSTEM",                _lc._AGENT6_SYSTEM,                 AGENT6_SYSTEM),
    ("RELEVANCE_SYSTEM",             _lc._RELEVANCE_SYSTEM,              RELEVANCE_SYSTEM),
    ('AGENT5_AUDIENCES["expert"]',   _lc._AGENT5_AUDIENCES["expert"],    AGENT5_AUDIENCES["expert"]),
    ('AGENT5_AUDIENCES["doctor"]',   _lc._AGENT5_AUDIENCES["doctor"],    AGENT5_AUDIENCES["doctor"]),
    ('AGENT5_AUDIENCES["layperson"]', _lc._AGENT5_AUDIENCES["layperson"], AGENT5_AUDIENCES["layperson"]),
]

print("Prompt diff vs tools/llm_conversation.py:\n")
for name, src, nb in PROMPTS_TO_DIFF:
    if src.strip() == nb.strip():
        print(f"  {name}: unchanged")
        continue
    print(f"  {name}: CHANGED ↓")
    diff = difflib.unified_diff(
        src.splitlines(),
        nb.splitlines(),
        fromfile=f"tools/llm_conversation.py:{name}",
        tofile=f"notebook:{name}",
        lineterm="",
    )
    for line in diff:
        print("    " + line)
    print()